In [1]:
import json
import numpy as np
import pandas as pd


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Prepare

In [3]:
train_df = pd.read_json('/content/drive/MyDrive/archive/data_for_train.jsonl', lines = True)
test_df = pd.read_json('/content/drive/MyDrive/archive/data_for_eval.jsonl', lines = True)

In [49]:

train_df.head(5)


,Text,address,name,normalized_main_rubric_name_ru,permalink,prices_summarized,relevance,reviews_summarized,relevance_new
0,Стриптиз клубы ижевск,"Удмуртская Республика, Ижевск, Береговая улица",More,Яхт-клуб,184649161205,"Яхт-клуб More предлагает прогулки на яхтах, ар...",0.0,Общий обзор отзывов: яхт-клуб «More» в Ижевске...,0.0
1,Подшипник KAMAZ 45105000000000,"Ростовская область, Таганрог, улица Ломоносова...","Подшипник; Kompaniya Podshipnik; Подшипник, ТД...",Магазин автозапчастей и автотоваров,84348429597,None,0.1,Организация занимается продажей автозапчастей ...,0.1
2,такелажные компании,"Киров, улица Менделеева, 2",Кировская такелажная компания; Kirovskaya Take...,Строительные леса,1095265306,Кировская такелажная компания предлагает строп...,1.0,Организация занимается продажей строительных л...,1.0
3,баннеры,"Свердловская область, Екатеринбург, улица Карл...","Дабл, Онлайн-Полиграфия; Double Print; Студия ...",Полиграфические услуги,1221953140,None,1.0,Организация занимается полиграфическими услуга...,1.0
4,лучший ресторан москвы 2016,"Москва, улица Крымский Вал, 9с1",Сыроварня; Syrovarnya; Сыроварня в Парке Горьк...,Ресторан,191911335353,Ресторан предлагает разнообразные блюда: от за...,1.0,Организация занимается приготовлением и подаче...,1.0


In [5]:
print(len(train_df['reviews_summarized'][0]))

2213


In [6]:
train_df['relevance'].value_counts()

,count
relevance,
1.0,15455
0.0,14075
0.1,4564


In [7]:
#deepseek/deepseek-v4-flash
apikey = '###'
Base_url = "https://api.vsegpt.ru/v1"

# Baseline

In [8]:
!pip install -q langchain_openai

In [9]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
from sklearn.metrics import accuracy_score
from tqdm import tqdm

In [10]:
llm = ChatOpenAI(
    api_key = apikey,
    base_url = Base_url,
    model = 'deepseek/deepseek-v4-flash',
    max_tokens=20000,
    temperature = 0
)
class RelevanceOutput(BaseModel):
  explanation: str = Field(description = 'краткое объяснение, почему выбрана такая оценка')
  relevance: float = Field(description = 'Оценка релевантности: 1.0 (Relevant Plus), 0.1 (Relevant Minus), 0.0 (Irrelevant)')
parser = JsonOutputParser(pydantic_object = RelevanceOutput)



        "Пример 4:\n"
        "Назвиние организации:\n"
        "Основная рубрика:\n"
        "Адрес:\n"
        "Описание\n"
        "Сводка отзывов пользователей:\n"
        "Оценка: \n\n"

In [11]:
baseline_prompt_withoutfs = ChatPromptTemplate.from_messages([
    ("system", (
        "Ты — эксперт-асессор Яндекс Карт. Твоя задача — оценить, насколько организация релевантна поисковому запросу пользователя.\n"
        "Запрос пользователя может быть рубричным (например, 'ресторан с верандой').\n\n"
        "Шкала оценки:\n"
        "- 1.0 (RELEVANT_PLUS): Организация идеально подходит под запрос, этот признак подтвержден.\n"
        "- 0.1 (RELEVANT_MINUS): Организация относится к нужной сфере, но нет четкого подтверждения специфичного признака (или это сеть, где признак есть не везде).\n"
        "- 0.0 (IRRELEVANT): Организация вообще не подходит под запрос.\n\n"

        "Выведи ответ строго в формате JSON со следующими полями:\n"
        '{{"explanation": "твое обоснование", "relevance": 1.0 или 0.1 или 0.0}}\n'
        "Отвечай только валидным JSON, без лишнего текста вокруг."
    )),
    ("user", (
        "Поисковый запрос: {query}\n"
        "Информация об организации:\n{org_info}\n"
    ))
])
baseline_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "Ты — эксперт-асессор Яндекс.Карт. Твоя задача — оценить, насколько организация релевантна поисковому запросу пользователя.\n"
        "Запрос пользователя может быть рубричным (например, 'ресторан с верандой').\n\n"
        "Шкала оценки:\n"
        "- 1.0 (RELEVANT_PLUS): Организация идеально подходит под запрос, этот признак подтвержден.\n"
        "- 0.1 (RELEVANT_MINUS): Организация относится к нужной сфере, но нет четкого подтверждения специфичного признака.\n"
        "- 0.0 (IRRELEVANT): Организация вообще не подходит под запрос.\n\n"
        "Примеры оценок:\n"

        "Пример 1:\n"
        "Поисковый запрос: сигары\n"
        "Название организации: Tabaccos\n"
        "Основная рубрика: Магазин табака и курительных принадлежностей\n"
        "Адрес: Москва, Дубравная улица, 34/29\n"
        "Описание: Нет\n"
        "Сводка отзывов пользователей: Организация занимается продажей табака, курительных принадлежностей и сопутствующих товаров.\n"
        "Ответ:\n"
        '{{"explanation": "Запрос явно указывает товар, магазин специализируется именно на табаке и курительных принадлежностях – полное совпадение.", "relevance": 1.0}}\n\n'

        "Пример 2:\n"
        "Поисковый запрос: вегетарианские кафе в москве\n"
        "Название организации: Горыныч\n"
        "Основная рубрика: Ресторан\n"
        "Адрес: Москва, Рождественский бульвар, 1\n"
        "Описание: Ресторан «Горыныч» предлагает разнообразные блюда, но нет информации о вегетарианской кухне.\n"
        "Сводка отзывов пользователей: Организация «Горыныч» — это ресторан в Москве, отзывы о кухне не упоминают вегетарианских блюд.\n"
        "Ответ:\n"
        '{{"explanation": "Организация является рестораном (подходящая сфера), но нет упоминания вегетарианской кухни – специфический признак не подтверждён.", "relevance": 0.1}}\n\n'

        "Пример 3:\n"
        "Поисковый запрос: аэс\n"
        "Название организации: Гунибская ГЭС\n"
        "Основная рубрика: АЭС , ГЭС , ТЭС\n"
        "Адрес: Республика Дагестан, Гунибский район, сельское поселение\n"
        "Описание: Нет\n"
        "Сводка отзывов пользователей: Организация занимается эксплуатацией гидроэлектростанции.\n"
        "Ответ:\n"
        '{{"explanation": "Запрос про атомную электростанцию, но организация относится к гидроэнергетике – сферы деятельности не совпадают.", "relevance": 0.0}}\n\n'

        "Теперь оцени новый запрос по указанным примерам.\n"
        "Выведи ответ в формате JSON со следующими полями:\n"
        '{{"explanation": "твое обоснование", "relevance": 1.0 или 0.1 или 0.0}}\n'
        "Отвечай только валидным JSON, без лишнего текста вокруг."
    )),
    ("user", (
        "Поисковый запрос: {query}\n"
        "Информация об организации:\n{org_info}\n"
    ))
])
baseline_wfs = baseline_prompt_withoutfs | llm | parser
baseline_chain = baseline_prompt | llm | parser

In [12]:
def evaluate_pipeline(pipeline, dataframe, num_samples):
  preds = []
  targs = []

  test_samples = dataframe.sample(n = num_samples, random_state = 342)
  for _, row in tqdm(test_samples.iterrows(), total = num_samples):
    org_info = f"""
    Название организации: {row.get('name')}
    Основная рубрика: {row.get('normalized_main_rubric_name_ru', 'Нет')}
    Адрес: {row.get('address', 'Нет')}
    Описание: {row.get('prices_summarized', 'Нет')}
    Сводка отзывов пользователей: {row.get('reviews_summarized', 'Нет')}
    """
    query = row['Text']
    try:
      res = pipeline(query, org_info)
      pred = float(res['relevance'])
      if pred not in [0.0, 0.1, 1.0]:
        pred = min([0.0, 0.1, 1.0], key = lambda x: abs(x-pred))
    except Exception as e:
      print(f"обишка запрос '{query}'): {e}")
      pred = 0.0
    preds.append(pred)
    targs.append(float(row['relevance']))
  print(preds, targs)
  acc = np.mean(np.array(preds) == np.array(targs))
  print(acc)
  return acc

def run_basefs(query, org_info):
  return baseline_chain.invoke({'query': query, 'org_info': org_info})
def run_basews(query, org_info):
  return baseline_wfs.invoke({'query': query, 'org_info': org_info})

In [ ]:
bl_acc = evaluate_pipeline(run_basews, train_df, num_samples = 300)#без few-shot (выдало 0.41 на 100 примерах, 0.46333 на 300)

100%|██████████| 300/300 [13:37<00:00,  2.72s/it]

[1.0, 0.1, 0.1, 0.1, 1.0, 1.0, 0.1, 0.0, 0.0, 1.0, 0.1, 0.1, 1.0, 0.0, 0.1, 0.1, 1.0, 0.1, 0.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.1, 1.0, 0.1, 0.1, 1.0, 0.1, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.1, 1.0, 0.1, 0.1, 0.0, 0.1, 0.1, 0.1, 1.0, 0.1, 1.0, 0.1, 1.0, 0.1, 1.0, 1.0, 1.0, 0.1, 1.0, 0.1, 1.0, 0.0, 0.1, 0.1, 1.0, 1.0, 0.1, 0.1, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.1, 1.0, 0.1, 1.0, 1.0, 1.0, 0.1, 1.0, 1.0, 0.1, 0.1, 0.1, 1.0, 0.0, 1.0, 0.1, 1.0, 0.1, 0.0, 0.0, 0.1, 0.1, 0.1, 0.0, 0.1, 1.0, 0.1, 0.0, 0.1, 0.1, 0.1, 1.0, 0.1, 1.0, 1.0, 0.0, 0.0, 0.1, 0.0, 0.1, 1.0, 0.0, 1.0, 0.1, 0.1, 0.0, 0.0, 0.1, 0.0, 1.0, 0.0, 0.0, 1.0, 0.1, 0.1, 0.1, 1.0, 1.0, 0.0, 0.1, 0.1, 0.1, 0.1, 1.0, 0.1, 0.1, 0.0, 0.1, 1.0, 0.1, 0.1, 1.0, 1.0, 0.1, 0.0, 0.0, 1.0, 0.1, 0.1, 0.0, 0.1, 0.1, 0.1, 0.1, 0.0, 1.0, 1.0, 0.1, 1.0, 0.0, 1.0, 0.1, 0.0, 0.0, 0.1, 0.0, 0.0, 1.0, 0.1, 1.0, 1.0, 0.1, 1.0, 0.0, 1.0, 0.1, 0.1, 0.1, 0.1, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.1, 0.0, 1.0, 1.0, 0.1, 0.1, 0.1, 0.0, 0.1, 0.1,

In [ ]:
bl_accfs = evaluate_pipeline(run_basefs, train_df, num_samples = 100)# с few-shot (выдало 0.39)
#видимо fs в данном виде неуместен

100%|██████████| 100/100 [04:04<00:00,  2.44s/it]

[1.0, 0.1, 0.1, 0.1, 1.0, 1.0, 0.1, 0.0, 0.1, 1.0, 0.1, 0.1, 1.0, 0.0, 0.1, 0.1, 1.0, 0.1, 0.1, 0.1, 1.0, 0.0, 1.0, 0.1, 0.1, 1.0, 1.0, 0.1, 1.0, 0.1, 0.1, 1.0, 1.0, 0.1, 1.0, 0.0, 1.0, 0.0, 0.1, 0.1, 0.0, 0.1, 1.0, 0.1, 1.0, 0.0, 0.1, 0.1, 1.0, 1.0, 0.1, 1.0, 0.1, 1.0, 0.1, 1.0, 1.0, 1.0, 0.1, 1.0, 0.1, 1.0, 0.1, 0.1, 0.1, 1.0, 0.1, 0.1, 0.1, 1.0, 0.1, 1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.1, 1.0, 1.0, 0.1, 0.1, 0.1, 1.0, 0.0, 1.0, 0.1, 0.1, 0.1, 0.1, 0.0, 0.1, 0.0, 0.1, 0.1, 0.1] [1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.1, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.1, 0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.1, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.1, 0.0, 0.1, 1.0, 0.0, 0.0, 1.0, 0.1, 1.0, 0.0, 1.0, 0.0, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.1, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.1, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.1, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0

# AGENT

In [13]:
!pip install duckduckgo-search langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [15]:
!pip install -U ddgs langchain-community


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 2.9 MB/s eta 0:00:00


In [13]:
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.messages import SystemMessage
import re

/tmp/ipykernel_5980/2140625984.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


In [16]:
search_tool = DuckDuckGoSearchRun()
class AgentState(TypedDict):
  messages: Annotated[list, add_messages]
  query: str
  org_info: str
  final_output: dict
  tool_calls_count: int
llm_wt = llm.bind_tools([search_tool])

In [17]:
def call_model(state: AgentState):
    system_msg = SystemMessage(content=(
        "Ты - агент-исследователь для Яндекс Карт.\n"
        f"Оцени релевантность организации для запроса: '{state['query']}'\n"
        f"Данные из базы: \n{state['org_info']}\n\n"
        "Твоя задача - проверить, соответствует ли организация запросу.\n"
        "Если информации НЕДОСТАТОЧНО, ты ОБЯЗАН вызвать инструмент поиска 'DuckDuckGoSearchRun'.\n"
        "Ищи запросы вида: '[название организации] [город] [ключевое слово]'.\n"
        "Как только соберешь достаточно информации, просто напиши финальный вывод, инструмент финализации сам его отформатирует."
    ))

    messages = [system_msg] + state['messages']
    response = llm_wt.invoke(messages)
    return {"messages": [response]}

In [18]:
def execute_tool(state: AgentState):
  last_msg = state['messages'][-1]
  tool_msgs = []
  for tool_call in last_msg.tool_calls:
      try:
          query_arg = tool_call['args'].get('query', str(tool_call['args']))
          search_result = search_tool.invoke(query_arg)
      except Exception as e:
          search_result = f"error {e}"
      tool_msgs.append(
          ToolMessage(
              content = str(search_result),
              tool_call_id = tool_call['id']
          )
      )
  return {"messages": tool_msgs, "tool_calls_count": state['tool_calls_count'] + 1}

In [80]:
def should_continue(state: AgentState):
    last_msg = state['messages'][-1]
    if state.get('tool_calls_count', 0) >= 5:
        return 'end'
    if last_msg.tool_calls:
        return 'tool'
    return 'end'

# Вставка

In [20]:
#Решил сделать отдельный запрос для заключения вердикта по результатам поиска

In [21]:
def finalize(state: AgentState):
    structured_llm = llm.with_structured_output(RelevanceOutput, method="function_calling")
    search_results = [
        msg.content for msg in state['messages'] if isinstance(msg, ToolMessage)
    ]
    extra_info = "\n".join(search_results) if search_results else "Дополнительная информация не найдена."
    final_prompt = f"""Определи релевантность организации для запроса: '{state['query']}'.
    Данные организации из базы: {state['org_info']}
    Собранная дополнительная информация из интернета:
    {extra_info}
    """
    response = structured_llm.invoke(final_prompt)
    final_dict = response.model_dump() if hasattr(response, 'model_dump') else response

    return {"final_output": final_dict}


# Вставка

### graph



In [22]:
workflow = StateGraph(AgentState)
workflow.add_node('agent', call_model)
workflow.add_node('tool', execute_tool)
workflow.add_node('finalize', finalize)

workflow.set_entry_point('agent')
workflow.add_conditional_edges(
    'agent',
    should_continue,
    {
        'tool': 'tool',
        'end': 'finalize'
    }
)
workflow.add_edge('tool', 'agent')
workflow.add_edge('finalize', END)
app = workflow.compile()

### run func

In [23]:
def run_ag(query: str, org_info: str):
    initial_message = HumanMessage(content=f"Начни исследование для запроса: '{query}'")
    inputs = {
        'query': query,
        'org_info': org_info,
        'messages': [initial_message],
        'tool_calls_count': 0
    }
    config = {'recursion_limit': 15}
    result = app.invoke(inputs, config)
    return result['final_output']

In [ ]:
agent_acc = evaluate_pipeline(run_ag , train_df, num_samples = 50) #

100%|██████████| 50/50 [15:39<00:00, 18.79s/it]

[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.1, 1.0, 0.0, 1.0, 0.1, 1.0, 0.0, 0.1, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 0.1, 1.0, 1.0, 0.1, 1.0, 0.1, 1.0, 0.0, 0.0, 0.1, 0.1, 1.0, 1.0, 0.1, 1.0, 0.1, 1.0, 0.1, 1.0, 1.0] [1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.1, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.1, 0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.1, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.1, 0.0, 0.1, 1.0, 0.0, 0.0, 1.0]
0.6


# Inference -------------------------------------------------------------------------

In [24]:
test_df.head(5)

,Text,address,name,normalized_main_rubric_name_ru,permalink,prices_summarized,reviews_summarized,relevance_new
0,сигары,"Москва, Дубравная улица, 34/29",Tabaccos; Магазин Tabaccos; Табаккос,Магазин табака и курительных принадлежностей,1263329400,None,"Организация занимается продажей табака, курите...",1.0
1,кальянная спб мероприятия,"Санкт-Петербург, Большой проспект Петроградско...",PioNero; Pionero; Пицца Паста бар; Pio Nero; P...,Кафе,228111266197,PioNero предлагает разнообразные блюда итальян...,"Организация PioNero — это кафе, бар и ресторан...",0.0
2,Эпиляция,"Московская область, Одинцово, улица Маршала Жу...",MaxiLife; Центр красоты и здоровья MaxiLife; Ц...,Стоматологическая клиника,1247255817,"Стоматологическая клиника, массажный салон и к...","Организация занимается стоматологическими, кос...",1.0
3,спортзал где 1 занятие бесплатно,"Москва, Страстной бульвар, 13А",Унца Унца Спорт; Unza Unza Sport,Фитнес-клуб,201938477844,Фитнес-клуб предлагает пробные занятия по разл...,Организация «Унца Унца Спорт» предоставляет ус...,0.1
4,стиральных машин,"Москва, улица Обручева, 34/63",М.Видео; M Video; M. Видео; M.Видео; Mvideo; М...,Магазин бытовой техники,1074529324,М.Видео предлагает широкий ассортимент бытовой...,Организация занимается продажей бытовой техник...,1.0


In [33]:
test_df['relevance_new'].value_counts()

,count
relevance_new,
1.0,317
0.0,183
0.1,70


In [47]:
test_df_inf = test_df.drop(test_df[test_df['relevance_new'] == 0.1].index, inplace = True)# в чате тг что 0.1 быть не должно (т.к. они сильно бьют по качеству, я их выкину)
test_df_inf = test_df.rename(columns={'relevance_new': 'relevance'})# чтобы не переписывать функции выше поменял название целевой переменной

In [48]:
test_df_inf.head(5)

,Text,address,name,normalized_main_rubric_name_ru,permalink,prices_summarized,reviews_summarized,relevance
0,сигары,"Москва, Дубравная улица, 34/29",Tabaccos; Магазин Tabaccos; Табаккос,Магазин табака и курительных принадлежностей,1263329400,None,"Организация занимается продажей табака, курите...",1.0
1,кальянная спб мероприятия,"Санкт-Петербург, Большой проспект Петроградско...",PioNero; Pionero; Пицца Паста бар; Pio Nero; P...,Кафе,228111266197,PioNero предлагает разнообразные блюда итальян...,"Организация PioNero — это кафе, бар и ресторан...",0.0
2,Эпиляция,"Московская область, Одинцово, улица Маршала Жу...",MaxiLife; Центр красоты и здоровья MaxiLife; Ц...,Стоматологическая клиника,1247255817,"Стоматологическая клиника, массажный салон и к...","Организация занимается стоматологическими, кос...",1.0
4,стиральных машин,"Москва, улица Обручева, 34/63",М.Видео; M Video; M. Видео; M.Видео; Mvideo; М...,Магазин бытовой техники,1074529324,М.Видео предлагает широкий ассортимент бытовой...,Организация занимается продажей бытовой техник...,1.0
5,сеть быстрого питания,"Санкт-Петербург, 1-я Красноармейская улица, 15",Rostic's; KFC; Ресторан быстрого питания KFC,Быстрое питание,1219173871,Rostic's предлагает различные наборы быстрого ...,"Организация занимается быстрым питанием, предо...",1.0


In [78]:
baseline_prompt_withoutfs = ChatPromptTemplate.from_messages([
    ("system", (
        "Ты — эксперт-асессор Яндекс Карт. Твоя задача — оценить, насколько организация релевантна поисковому запросу пользователя.\n"
        "Запрос пользователя может быть рубричным (например, 'ресторан с верандой').\n\n"
        "Шкала оценки:\n"
        "- 1.0 (RELEVANT_PLUS): Организация подходит под запрос, этот признак подтвержден.\n"
        "- 0.0 (IRRELEVANT): Организация не подходит под запрос.\n\n"
        "Выведи ответ строго в формате JSON со следующими полями:\n"
        '{{"explanation": "твое обоснование", "relevance": 1.0 или 0.0}}\n'
        "Отвечай только валидным JSON, без лишнего текста вокруг."
    )),
    ("user", (
        "Поисковый запрос: {query}\n"
        "Информация об организации:\n{org_info}\n"
    ))
])# 0.1 удалены, поэтому 1.0 должна включать сэмплы, которые должны были отнестить к 0.1
class RelevanceOutput(BaseModel):
  explanation: str = Field(description = 'краткое объяснение, почему выбрана такая оценка')
  relevance: float = Field(description = 'Оценка релевантности: 1.0 (Relevant Plus), 0.0 (Irrelevant)')
parser = JsonOutputParser(pydantic_object = RelevanceOutput)

In [76]:
def evaluate_pipeline(pipeline, dataframe, num_samples):
  preds = []
  targs = []

  test_samples = dataframe.sample(n = num_samples, random_state = 342)
  for _, row in tqdm(test_samples.iterrows(), total = num_samples):
    org_info = f"""
    Название организации: {row.get('name')}
    Основная рубрика: {row.get('normalized_main_rubric_name_ru', 'Нет')}
    Адрес: {row.get('address', 'Нет')}
    Описание: {row.get('prices_summarized', 'Нет')}
    Сводка отзывов пользователей: {row.get('reviews_summarized', 'Нет')}
    """
    query = row['Text']
    try:
      res = pipeline(query, org_info)
      pred = float(res['relevance'])
      if pred not in [0.0, 1.0]:
        pred = min([0.0, 1.0], key = lambda x: abs(x-pred))
    except Exception as e:
      print(f"обишка запрос '{query}'): {e}")
      pred = 0.0
    preds.append(pred)
    targs.append(float(row['relevance']))
  print(preds, targs)
  acc = np.mean(np.array(preds) == np.array(targs))
  print(acc)
  return acc

def run_basefs(query, org_info):
  return baseline_chain.invoke({'query': query, 'org_info': org_info})
def run_basews(query, org_info):
  return baseline_wfs.invoke({'query': query, 'org_info': org_info})

In [67]:
test = test_df_inf.head(30)

# Test run

In [79]:
bl_acc = evaluate_pipeline(run_basews, test, num_samples = 30)#без fs примеров

100%|██████████| 30/30 [01:28<00:00,  2.96s/it]

[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0] [1.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 1.0]
0.6


In [81]:
ag_acc = evaluate_pipeline(run_ag, test, num_samples = 30)# сразу видим, что появилось гораздо больше единиц, значит агент актуален, притом модель начала ошибаться в сэмплах с меткой 1.0,
#а значит это скорее всего можно частично пофиксить промтом

100%|██████████| 30/30 [10:35<00:00, 21.20s/it]

[1.0, 1.0, 0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0] [1.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 1.0]
0.8


# Result

In [83]:
ag_acc_full = evaluate_pipeline(run_ag, test_df_inf, num_samples= 500)# agent

100%|██████████| 500/500 [2:54:20<00:00, 20.92s/it]

[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 0.0, 1.0, 1.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0,

In [84]:
bl_acc_full = evaluate_pipeline(run_basews, test_df_inf, num_samples= 500)# llm

100%|██████████| 500/500 [25:49<00:00,  3.10s/it]

[1.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0,

In [ ]:
#у агента получился результат 0.794, у llm 0.662